# 01 — Data Audit (Phase 1)

Loads the parquet-cached challenge tables and reports the required audit
statistics. The heavy scan over the full ~2.2M/5M/5.3M-row tables was run
once via `cache/audit_scratch.py` (streaming, column-pruned reads through
`src.data_loader`) and cached to `experiments/data_audit_report.json`; this
notebook loads that cache so re-running it (e.g. to regenerate this
notebook) is instant rather than re-scanning the raw data every time.

To regenerate the underlying numbers from scratch: `python cache/audit_scratch.py`.


In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src import config

report = json.loads((config.EXPERIMENTS_DIR / "data_audit_report.json").read_text())
list(report.keys())


['train_source1',
 'train_source2',
 'train_source3',
 'test_source1',
 'test_source2',
 'test_source3',
 'ground_truth']

## Row counts, dtypes, and basic quality per table

In [2]:
rows = []
for key in ["train_source1", "train_source2", "train_source3", "test_source1", "test_source2", "test_source3"]:
    s = report[key]
    rows.append({
        "table": key,
        "n_rows": s["n_rows"],
        "n_unique_entity_id": s["n_unique_entity_id"],
        "dup_entity_id_rows": s["n_duplicate_entity_id_rows"],
        "n_unique_name": s["n_unique_business_name"],
        "n_unique_address": s["n_unique_business_address"],
        "exact_name_addr_dup_rows": s["n_exact_name_addr_duplicate_rows"],
        "pct_empty_name": round(s["pct_empty_name"], 3),
        "pct_empty_address": round(s["pct_empty_address"], 3),
        "name_len_mean": round(s["name_len_mean"], 1),
        "addr_len_mean": round(s["addr_len_mean"], 1),
    })
pd.DataFrame(rows)


,table,n_rows,n_unique_entity_id,dup_entity_id_rows,n_unique_name,n_unique_address,exact_name_addr_dup_rows,pct_empty_name,pct_empty_address,name_len_mean,addr_len_mean
0,train_source1,2206821,2206821,0,1539229,2130606,0,0.0,0.000,24.0,52.1
1,train_source2,5034616,5034616,0,4402009,4337262,50969,0.0,3.356,25.1,46.2
2,train_source3,5285603,5285603,0,4651609,4632765,37283,0.0,3.328,25.2,46.7
3,test_source1,1732544,1732544,0,1238867,1677483,0,0.0,0.000,23.8,57.2
4,test_source2,4887273,4887273,0,4311041,4224784,44552,0.0,2.648,25.7,50.4
5,test_source3,5082316,5082316,0,4521929,4456436,32178,0.0,2.678,25.7,48.7


**Observations**

* `entity_id` is unique within every table (0 duplicate ids anywhere) --
  there is no accidental row duplication to clean up structurally.
* `business_name` is never empty in any table; `business_address` is empty
  in ~2.6-3.4% of Source-2/Source-3 rows (never in Source-1). Address
  features must tolerate missing addresses gracefully (see
  `normalization.py` / `features.py`: an empty address never crashes a
  similarity computation, and `addr_both_present` is an explicit feature).
* Exact full-row (name+address) duplicates exist in Source-2/Source-3
  (~1% of rows) but not in Source-1, consistent with Source-1 being the
  *deduplicated* reference source described in the problem statement.
* Business names repeat much more than they are unique (e.g. train Source-1:
  1,539,229 unique names out of 2,206,821 rows, ~70%) -- common business
  names recur across different real-world entities, so name-only blocking
  keys would create very large, low-precision blocks. This directly
  motivates combining name blocks with address/country signals (see
  `blocking.py`).


## Country distribution — open-set, France appears only in test

In [3]:
for key in ["train_source1", "test_source1"]:
    print(key, report[key]["country_counts"])


train_source1 {'US': 1323633, 'India': 883188}
test_source1 {'India': 809986, 'US': 663106, 'France': 259452}


Training only ever sees `US` and `India`. The test set additionally
contains `France` (~15% of test Source-1 rows) with **zero** training
examples. This is exactly why `normalization.py` and `blocking.py` never
branch on a fixed country list -- `country` is only ever used as an
open-set string feature/blocking key (equality, missingness), never as a
hard-coded filter.

## Ground truth: match-count distribution, singleton rate, fan-in check

In [4]:
gt = report["ground_truth"]
print(f"Source-1 training entities: {gt['n_s1_entities']:,}")
print(f"  zero matches (singletons): {gt['n_zero_match']:,} ({gt['pct_zero_match']:.2f}%)")
print(f"  exactly one match:         {gt['n_one_match']:,}")
print(f"  multiple matches:          {gt['n_multi_match']:,}")
print(f"Total true (S1, matched_id) pairs: {gt['total_true_pairs']:,}")
print(f"  of which -> Source-2: {gt['total_s2_true_pairs']:,}, -> Source-3: {gt['total_s3_true_pairs']:,}")
print()
print("Match-count distribution (n_matches -> n_s1_entities):")
for k, v in gt["match_count_dist"].items():
    print(f"  {k}: {v:,}")


Source-1 training entities: 2,206,821
  zero matches (singletons): 123,247 (5.58%)
  exactly one match:         119,157
  multiple matches:          1,964,417
Total true (S1, matched_id) pairs: 7,638,365
  of which -> Source-2: 3,693,619, -> Source-3: 3,944,746

Match-count distribution (n_matches -> n_s1_entities):
  0: 123,247
  1: 119,157
  2: 375,212
  3: 530,841
  4: 484,115
  5: 321,957
  6: 164,868
  7: 63,968
  8: 18,680
  9: 4,205
  10: 534
  11: 37


In [5]:
print(f"Matched ids (S2/S3) claimed by >1 Source-1 parent: {gt['n_matched_ids_with_multiple_parents']}")
print(f"Max parents seen for any matched id: {gt['max_parents_per_matched_id']}")


Matched ids (S2/S3) claimed by >1 Source-1 parent: 0
Max parents seen for any matched id: 1


**Key findings that shape the rest of the pipeline:**

1. **~5.6% of Source-1 training entities are true singletons** (no match in
   Source-2/3). Since F0.5 rewards correctly predicting empty (1.0) and
   punishes any false merge on a singleton (0.0), the decision threshold
   must be tuned with this in mind, not just for raw pair accuracy (Phase 8).
2. Non-singleton entities average **~3.7 true matches each** (mostly 2-5) --
   Source-2 and Source-3 both contain multiple noisy duplicate records per
   real business, so the matcher must support **one-to-many** Source-1 ->
   {Source-2, Source-3} matching, not a 1:1 assignment.
3. **No Source-2/Source-3 id is ever claimed by more than one Source-1
   parent** (0 out of 7,638,365 true pairs violate this, checked exhaustively).
   This is a strong, dataset-wide consistency signal:
   `thresholding.resolve_one_parent_per_match` implements an optional
   post-processing rule (keep only the highest-scoring assignment when a
   candidate is accepted by multiple Source-1 queries) that trades a little
   recall for precision -- to be validated empirically (Phase 18) rather
   than assumed to help.
